# Facility Location on Graphs

This notebook demonstrates the uncapacitated facility location problem on graphs. We:

- Build a small graph (random geometric / Erdős–Rényi examples)
- Compute shortest-path distances between nodes
- Define opening costs for facilities located at graph nodes
- Solve for the optimal subset of facilities by brute force (small graphs)
- Compare with a simple greedy heuristic
- Visualize the assignment of clients to facilities

Problem statement (informal): Given a graph whose nodes are both potential facilities and clients, choose a set of facilities to open. Opening a facility incurs a fixed cost; each client is assigned to the nearest open facility and pays the distance cost. The objective is to minimize total opening + assignment cost.

In [1]:
import itertools
import math
import random
import time

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

plt.rcParams['figure.figsize'] = (7, 5)
np.set_printoptions(precision=3, suppress=True)


## Utilities: distance matrix, cost evaluation, visualization

We treat every node as both a client and a candidate facility. Opening costs are provided per-node (can be random or constant).

In [2]:
def compute_distance_matrix(G, weight=None):
    """Compute all-pairs shortest-path distances as a numpy array.
    Returns (nodes_list, dist_matrix) where dist_matrix[i,j] is distance from nodes[i] to nodes[j].
    """
    nodes = list(G.nodes())
    n = len(nodes)
    index = {v: i for i, v in enumerate(nodes)}
    D = np.full((n, n), np.inf)
    for u in nodes:
        # networkx returns dict of distances from u
        lengths = nx.single_source_dijkstra_path_length(G, u, weight=weight)
        for v, d in lengths.items():
            D[index[u], index[v]] = d
    return nodes, D

def facility_location_cost(open_set, open_costs, D, clients=None):
    """Total cost = sum(open_costs for open facilities) + sum(distance to nearest open facility for each client).
    open_set: iterable of facility indices (indices into D)
    open_costs: array-like of length n
    D: distance matrix (n x n) where D[i,j] is distance from i to j
    clients: iterable of client indices (defaults to all nodes)
    """
    n = D.shape[0]
    if clients is None:
        clients = range(n)
    open_set = set(open_set)
    if not open_set:
        return math.inf
    opening = sum(open_costs[i] for i in open_set)
    # for each client, compute distance to nearest open facility
    assign_cost = 0.0
    open_list = list(open_set)
    for c in clients:
        # use numpy indexing to get a 1D array of distances to open facilities
        # this avoids creating a Python list of possibly-array elements
        dists = D[c, open_list]
        best = float(np.min(dists))
        assign_cost += best
    return opening + assign_cost

def brute_force_optimal(open_costs, D, clients=None, max_size=None):
    """Brute force search for optimal open facilities. Only feasible for n<=15 or so.
    Returns (best_set, best_cost, elapsed_seconds)
    """
    n = D.shape[0]
    if clients is None:
        clients = range(n)
    best_cost = math.inf
    best_set = None
    start = time.time()
    # iterate all non-empty subsets (avoid empty because assignment cost infinite)
    for r in range(1, n + 1):
        if max_size is not None and r > max_size:
            break
        for comb in itertools.combinations(range(n), r):
            c = facility_location_cost(comb, open_costs, D, clients)
            if c < best_cost:
                best_cost = c
                best_set = set(comb)
    return best_set, best_cost, time.time() - start

def greedy_additive(open_costs, D, clients=None):
    """A simple greedy algorithm: start with empty set, repeatedly add the facility
    that yields the largest decrease in total cost.
    This is not guaranteed optimal but is fast.
    Returns (open_set, cost, history)
    """
    n = D.shape[0]
    if clients is None:
        clients = list(range(n))
    open_set = set()
    current_cost = math.inf
    history = []
    # We need a base assignment cost when no facility exists -> inf, so first pick must open something
    # We'll instead compute best single-facility choice, then iteratively add
    best_single = None
    best_single_cost = math.inf
    for i in range(n):
        c = facility_location_cost([i], open_costs, D, clients)
        if c < best_single_cost:
            best_single_cost = c
            best_single = i
    # If all single-facility costs are infinite (e.g., disconnected graph), pick
    # a sensible starting facility by penalizing unreachable distances so we
    # never insert None into the open_set.
    if best_single is None:
        finite_mask = np.isfinite(D)
        if np.any(finite_mask):
            big = np.max(D[finite_mask]) * 10.0
        else:
            big = 1e6
        best_score = math.inf
        for i in range(n):
            dists = D[:, i]
            total = np.sum(np.where(np.isfinite(dists), dists, big))
            score = open_costs[i] + float(total)
            if score < best_score:
                best_score = score
                best_single = i
        best_single_cost = facility_location_cost([best_single], open_costs, D, clients)
    open_set.add(best_single)
    current_cost = best_single_cost
    history.append((set(open_set), current_cost))
    while True:
        best_improvement = 0.0
        best_choice = None
        for j in range(n):
            if j in open_set:
                continue
            c = facility_location_cost(open_set | {j}, open_costs, D, clients)
            improvement = current_cost - c
            if improvement > best_improvement:
                best_improvement = improvement
                best_choice = j
        if best_choice is None:
            break
        open_set.add(best_choice)
        current_cost = facility_location_cost(open_set, open_costs, D, clients)
        history.append((set(open_set), current_cost))
    return open_set, current_cost, history

def assign_clients(open_set, D, nodes=None):
    """Return mapping client_index -> assigned facility index and distances."""
    n = D.shape[0]
    if nodes is None:
        nodes = list(range(n))
    assignments = {}
    open_list = list(open_set)
    for c in range(n):
        # distances to available facilities as a numpy array
        dists = D[c, open_list]
        idx = int(np.argmin(dists))
        best_f = open_list[idx]
        assignments[c] = (best_f, float(dists[idx]))
    return assignments

def plot_assignment(G, nodes, pos, open_set, assignments, title=None):
    """Visualize graph with facilities and client assignments. nodes is list mapping indices->node ids."""
    # color by assigned facility
    n = len(nodes)
    # choose a color per facility
    colors = plt.cm.tab10.colors
    facility_list = sorted(open_set)
    color_map = {f: colors[i % len(colors)] for i, f in enumerate(facility_list)}
    node_colors = []
    node_sizes = []
    labels = {}
    for i, v in enumerate(nodes):
        f, d = assignments[i]
        node_colors.append(color_map[f])
        if i in open_set:
            node_sizes.append(350)
            labels[v] = f"{v}\n(open)"
        else:
            node_sizes.append(150)
            labels[v] = f"{v}\n{d:.1f}"
    plt.figure()
    nx.draw_networkx_edges(G, pos=pos, alpha=0.4)
    nx.draw_networkx_nodes(G, pos=pos, nodelist=nodes, node_color=node_colors, node_size=node_sizes)
    nx.draw_networkx_labels(G, pos=pos, labels=labels, font_size=8)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()


## Example 1: Small random geometric graph

We'll create a random geometric graph where nodes are points in the unit square and edges connect nearby points. Distances are the Euclidean length of the shortest path on the graph (so edge lengths equal Euclidean distances).

In [3]:
random.seed(2)
n = 10
radius = 0.4
G = nx.random_geometric_graph(n, radius, seed=2)
pos = nx.get_node_attributes(G, 'pos')
if len(pos) != n:
    # ensure positions for all
    pos = {i: (random.random(), random.random()) for i in G.nodes()}

# set Euclidean length as edge weight
for u, v in list(G.edges()):
    px, py = pos[u]
    qx, qy = pos[v]
    d = math.hypot(px - qx, py - qy)
    G[u][v]['weight'] = d

nodes, D = compute_distance_matrix(G, weight='weight')
n = len(nodes)

# Opening costs: vary per node (e.g., random in [0.5, 2.0])
open_costs = np.round(np.random.uniform(0.5, 2.0, size=n), 2)
print('nodes:', nodes)
print('open_costs:', open_costs)
print('distance matrix:\n', D)


nodes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
open_costs: [0.97 1.59 1.95 0.56 1.99 1.5  1.33 0.85 1.52 1.71]
distance matrix:
 [[0.      inf 0.244 0.813 0.507 0.987 0.783 0.238 0.467 0.882]
 [  inf 0.      inf   inf   inf   inf   inf   inf   inf   inf]
 [0.244   inf 0.    0.569 0.263 0.743 0.539 0.282 0.223 0.638]
 [0.813   inf 0.569 0.    0.305 0.174 0.254 0.851 0.366 0.228]
 [0.507   inf 0.263 0.305 0.    0.479 0.276 0.546 0.349 0.375]
 [0.987   inf 0.743 0.174 0.479 0.    0.279 1.025 0.54  0.175]
 [0.783   inf 0.539 0.254 0.276 0.279 0.    0.822 0.62  0.126]
 [0.238   inf 0.282 0.851 0.546 1.025 0.822 0.    0.505 0.921]
 [0.467   inf 0.223 0.366 0.349 0.54  0.62  0.505 0.    0.594]
 [0.882   inf 0.638 0.228 0.375 0.175 0.126 0.921 0.594 0.   ]]


Run brute-force optimal search (feasible here because n=10).

In [4]:
best_set, best_cost, elapsed = brute_force_optimal(open_costs, D)
print(f'Brute-force optimal set indices: {sorted(best_set)} cost={best_cost:.3f} (t={elapsed:.3f}s)')

greedy_set, greedy_cost, history = greedy_additive(open_costs, D)
print(f'Greedy set indices: {sorted(greedy_set)} cost={greedy_cost:.3f}')

# visualise assignments for both
assign_opt = assign_clients(best_set, D, nodes)
plot_assignment(G, nodes, pos, best_set, assign_opt, title=f'Optimal (cost={best_cost:.2f})')

assign_greedy = assign_clients(greedy_set, D, nodes)
plot_assignment(G, nodes, pos, greedy_set, assign_greedy, title=f'Greedy (cost={greedy_cost:.2f})')


Brute-force optimal set indices: [1, 3, 7] cost=4.848 (t=0.102s)


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

## Notes and extensions

- The brute-force search is exponential in the number of nodes. For n > ~15 it's impractical.
- The greedy strategy implemented is a simple additive heuristic. Many more sophisticated heuristics and local-search methods exist for the facility location problem (e.g., primal-dual approximation algorithms, local search with swaps, LP relaxations + rounding, and mixed-integer programming) that achieve provable approximation guarantees.
- You can adapt this notebook to include facility capacities, different cost functions, or restrict candidate facility locations.
- If you want an exact solver on larger instances, try modeling the problem as an integer program and use a solver (e.g., CBC, Gurobi, or CPLEX) via pulp or gurobipy. That requires additional dependencies.

If you'd like, I can add:
1. A local-search (1-swap) improvement step to the greedy heuristic
2. An LP/MIP formulation using pulp and an example run (if you want to install pulp)
3. Benchmarking on larger graphs and a runtime/quality comparison